# Dataset Information

## Glass Identification Dataset

Conjunto de dados de classificação multiclasse com 214 amostras de vidro, cada uma descrita por seu índice de refração e pela composição química. O objetivo é prever o tipo de vidro a partir dessas medidas.

O problema vem da ciência forense: vidro deixado em uma cena de crime pode servir de evidência, desde que seja identificado corretamente. Saber se um fragmento veio de uma janela de prédio, de um farol de carro ou de um recipiente ajuda a ligar suspeitos ao local.

Características gerais
Item	Valor
Amostras	214
Atributos preditivos	9 (todos numéricos contínuos)
Atributo-alvo	Tipo de vidro (6 classes presentes)
Valores ausentes	Nenhum
Tarefa	Classificação
Atributos
##	Atributo	Descrição
1.	    Id	Identificador da amostra (1 a 214).
2.	    RI	Índice de refração
3.	    Na	Sódio
4.	    Mg	Magnésio
5.	    Al	Alumínio
6.	    Si	Silício
7.	    K	Potássio
8.	    Ca	Cálcio
9.	    Ba	Bário
10.	    Fe	Ferro
11.	    Type	Tipo de vidro (classe-alvo)

Os atributos de 3 a 10 são dados em percentual em peso do óxido correspondente (por exemplo, o Na representa Na₂O).

## Classe Alvo
Esta pode ser de 7 tipos:

Código  |   Tipo de vidro   |   Amostras
1.	    Janela de construção, float	
2.	    Janela de construção, não float	
3.	    Janela de veículo, float	
4.	    Janela de veículo, não float	
5.	    Recipientes	
6.	    Louça de mesa	
7. 	    Faróis	

Fonte: https://archive.ics.uci.edu/dataset/42/glass+identification


## importanto o dataset

In [5]:
from ucimlrepo import fetch_ucirepo 
import pandas as pd
  
# fetch dataset 
glass_identification = fetch_ucirepo(id=42) 
  
# data (as pandas dataframes) 
X = glass_identification.data.features 
y = glass_identification.data.targets 
  
# metadata 
#print(glass_identification.metadata) 
  
# variable information 
#print(glass_identification.variables) 

display(X)
display(y)


,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe
0,1.52101,13.64,4.49,1.10,71.78,0.06,8.75,0.00,0.0
1,1.51761,13.89,3.60,1.36,72.73,0.48,7.83,0.00,0.0
2,1.51618,13.53,3.55,1.54,72.99,0.39,7.78,0.00,0.0
3,1.51766,13.21,3.69,1.29,72.61,0.57,8.22,0.00,0.0
4,1.51742,13.27,3.62,1.24,73.08,0.55,8.07,0.00,0.0
...,...,...,...,...,...,...,...,...,...
209,1.51623,14.14,0.00,2.88,72.61,0.08,9.18,1.06,0.0
210,1.51685,14.92,0.00,1.99,73.06,0.00,8.40,1.59,0.0
211,1.52065,14.36,0.00,2.02,73.42,0.00,8.44,1.64,0.0
212,1.51651,14.38,0.00,1.94,73.61,0.00,8.48,1.57,0.0


,Type_of_glass
0,1
1,1
2,1
3,1
4,1
...,...
209,7
210,7
211,7
212,7


## Partição dos dados em treino e teste

### Holdout 80/20

In [7]:
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split

glass = fetch_ucirepo(id=42)
X, y = glass.data.features, glass.data.targets.squeeze()

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)


## Árvore de Decisão

In [11]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

grade = {"max_depth": [3, 4, 5, None], "min_samples_leaf": [1, 2, 4], "class_weight": [None, "balanced"]}
modelo = GridSearchCV(DecisionTreeClassifier(random_state=42), grade, cv=5, scoring="f1_macro").fit(X_tr, y_tr)

print(modelo.best_params_)
print(classification_report(y_te, modelo.predict(X_te), zero_division=0))


import graphviz
from sklearn.tree import export_graphviz

arvore = modelo.best_estimator_
dot = export_graphviz(arvore, feature_names=X.columns, class_names=[str(c) for c in arvore.classes_],
                      filled=True, rounded=True, impurity=False, label="none", special_characters=True)
g = graphviz.Source(dot.replace("digraph Tree {", "digraph Tree {\nrankdir=LR;", 1))

g.format = "svg"
g.render("arvore", cleanup=True)   # salva arvore.svg 
 


{'class_weight': 'balanced', 'max_depth': 4, 'min_samples_leaf': 1}
              precision    recall  f1-score   support

           1       0.75      0.43      0.55        14
           2       0.62      0.33      0.43        15
           3       0.25      1.00      0.40         3
           5       0.50      0.67      0.57         3
           6       0.67      1.00      0.80         2
           7       0.50      0.67      0.57         6

    accuracy                           0.51        43
   macro avg       0.55      0.68      0.55        43
weighted avg       0.62      0.51      0.51        43



'arvore.svg'

## Naive Bayes

In [13]:
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split

glass = fetch_ucirepo(id=42)

X = glass.data.features
y = glass.data.targets.squeeze()

X_tr, X_te, y_tr, y_te = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [14]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report

modelo_nb = GaussianNB().fit(X_tr, y_tr)

print(classification_report(
    y_te,
    modelo_nb.predict(X_te),
    zero_division=0
))

              precision    recall  f1-score   support

           1       0.50      0.86      0.63        14
           2       0.40      0.13      0.20        15
           3       0.00      0.00      0.00         3
           5       0.50      0.33      0.40         3
           6       0.50      1.00      0.67         2
           7       0.83      0.83      0.83         6

    accuracy                           0.51        43
   macro avg       0.46      0.53      0.46        43
weighted avg       0.48      0.51      0.45        43



In [15]:


print("Probabilidades das classes:")
display(pd.DataFrame({
    "classe": modelo_nb.classes_,
    "probabilidade": modelo_nb.class_prior_
}))

print("Médias das features por classe:")
display(pd.DataFrame(
    modelo_nb.theta_,
    index=modelo_nb.classes_,
    columns=X.columns
).round(3))

Probabilidades das classes:


,classe,probabilidade
0,1,0.327485
1,2,0.356725
2,3,0.081871
3,5,0.058480
4,6,0.040936
5,7,0.134503


Médias das features por classe:


,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe
1,1.519,13.201,3.558,1.156,72.633,0.460,8.816,0.004,0.066
2,1.519,13.058,3.100,1.410,72.584,0.533,9.021,0.061,0.086
3,1.518,13.439,3.545,1.222,72.324,0.412,8.806,0.011,0.062
5,1.519,12.711,1.006,2.026,72.478,1.210,10.086,0.244,0.079
6,1.518,14.761,1.117,1.300,73.091,0.000,9.614,0.000,0.000
7,1.518,14.457,0.540,2.087,72.942,0.325,8.536,1.035,0.013


In [16]:
amostra = X_te.iloc[[0]]

print("Classe real:", y_te.iloc[0])
print("Classe prevista:", modelo_nb.predict(amostra)[0])

display(pd.DataFrame({
    "classe": modelo_nb.classes_,
    "probabilidade": modelo_nb.predict_proba(amostra)[0]
}).sort_values("probabilidade", ascending=False))

Classe real: 7
Classe prevista: 7


,classe,probabilidade
5,7,9.990614e-01
3,5,9.385789e-04
1,2,1.781342e-08
2,3,5.559860e-143
0,1,2.639372e-230
4,6,0.000000e+00


## KNN

In [23]:
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split

glass = fetch_ucirepo(id=42)
X = glass.data.features
y = glass.data.targets.squeeze()

X_tr, X_te, y_tr, y_te = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [24]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import classification_report

pipe = Pipeline([("sc", StandardScaler()), ("knn", KNeighborsClassifier())])
grade = {"knn__n_neighbors": [1, 3, 5, 7, 9, 11, 15], "knn__weights": ["uniform", "distance"], "knn__p": [1, 2]}
busca = GridSearchCV(pipe, grade, cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring="f1_macro")
modelo = busca.fit(X_tr, y_tr)

print(modelo.best_params_)
print(classification_report(y_te, modelo.predict(X_te), zero_division=0))

{'knn__n_neighbors': 1, 'knn__p': 1, 'knn__weights': 'uniform'}
              precision    recall  f1-score   support

           1       0.75      0.64      0.69        14
           2       0.77      0.67      0.71        15
           3       0.40      0.67      0.50         3
           5       0.75      1.00      0.86         3
           6       0.67      1.00      0.80         2
           7       1.00      1.00      1.00         6

    accuracy                           0.74        43
   macro avg       0.72      0.83      0.76        43
weighted avg       0.76      0.74      0.75        43



In [25]:
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import confusion_matrix

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
r = cross_validate(busca, X, y, cv=cv, scoring=["accuracy", "f1_macro"], n_jobs=-1)
print(*[f"{m}: {r['test_'+m].mean():.3f} ± {r['test_'+m].std():.3f}" for m in ["accuracy", "f1_macro"]])

pred = cross_val_predict(busca, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=42), n_jobs=-1)
print(classification_report(y, pred, zero_division=0))
print(confusion_matrix(y, pred))

accuracy: 0.729 ± 0.057 f1_macro: 0.694 ± 0.081
              precision    recall  f1-score   support

           1       0.69      0.81      0.75        70
           2       0.77      0.70      0.73        76
           3       0.46      0.35      0.40        17
           5       0.57      0.62      0.59        13
           6       0.80      0.89      0.84         9
           7       0.96      0.83      0.89        29

    accuracy                           0.73       214
   macro avg       0.71      0.70      0.70       214
weighted avg       0.73      0.73      0.73       214

[[57  8  5  0  0  0]
 [16 53  2  4  1  0]
 [ 9  2  6  0  0  0]
 [ 0  4  0  8  0  1]
 [ 0  1  0  0  8  0]
 [ 1  1  0  2  1 24]]


In [26]:
# Interpretabilidade do KNN

melhor_knn = modelo.best_estimator_
amostra = X_te.iloc[[0]]

k = melhor_knn.named_steps["knn"].n_neighbors
peso = melhor_knn.named_steps["knn"].weights

p = melhor_knn.named_steps["knn"].p

amostra_escalada = melhor_knn.named_steps["sc"].transform(amostra)

distancias, indices = melhor_knn.named_steps["knn"].kneighbors(
    amostra_escalada,
    n_neighbors=k
)

classe_real = y_te.iloc[0]
classe_prevista = modelo.predict(amostra)[0]

print("Classe real:", classe_real)
print("Classe prevista:", classe_prevista)
print("Número de vizinhos (k):", k)
print("Tipo de peso:", peso)
print("Métrica p:", p)

print("\nAmostra analisada:")
print(amostra)

print("\nVizinhos mais próximos:")

for posicao, (indice, distancia) in enumerate(
    zip(indices[0], distancias[0]),
    start=1
):
    print(f"\nVizinho {posicao}")
    print("Índice no treino:", indice)
    print("Classe:", y_tr.iloc[indice])
    print("Distância:", distancia)
    print("Características:")
    print(X_tr.iloc[indice])

# Classes dos vizinhos
classes_vizinhos = y_tr.iloc[indices[0]]

print("\nClasses dos vizinhos:")
print(classes_vizinhos.values)

print("\nContagem das classes:")
print(classes_vizinhos.value_counts())

# Se estiver usando peso por distância, mostra também o peso de cada vizinho
if peso == "distance":
    pesos_vizinhos = 1 / (distancias[0] + 1e-10)

    print("\nPesos dos vizinhos:")
    for i, peso_vizinho in enumerate(pesos_vizinhos, start=1):
        print(f"Vizinho {i}: {peso_vizinho}")

# Resultado final
if classe_real == classe_prevista:
    print("\nResultado: previsão correta.")
else:
    print("\nResultado: previsão incorreta.")

Classe real: 7
Classe prevista: 7
Número de vizinhos (k): 1
Tipo de peso: uniform
Métrica p: 1

Amostra analisada:
         RI     Na   Mg    Al     Si    K    Ca    Ba   Fe
208  1.5164  14.37  0.0  2.74  72.85  0.0  9.45  0.54  0.0

Vizinhos mais próximos:

Vizinho 1
Índice no treino: 166
Classe: 7
Distância: 1.3854303585425982
Características:
RI     1.51531
Na    14.38000
Mg     0.00000
Al     2.66000
Si    73.10000
K      0.04000
Ca     9.08000
Ba     0.64000
Fe     0.00000
Name: 198, dtype: float64

Classes dos vizinhos:
[7]

Contagem das classes:
Type_of_glass
7    1
Name: count, dtype: int64

Resultado: previsão correta.
